# Orthogonal matrices - Rotations in 3D

By using consecutive rotations about the $x$, $y$, and $z$-axes in 3D, we can produce any orthogonal axis system (with the same orientation, i.e., handedness as the original axes.)

This notebook contains an interactive visualization of 

$$
    R = R_z(\gamma) R_y(\beta) R_z(\alpha)
$$

where $\alpha$, $\beta$, and $\gamma$ are angles, and $R_i(\theta)$ rotates about axis $i$ with angle $\theta$.

The new rotated axes are:
$$
\mathbf e'_i = R\,\mathbf e_i, \quad i \in \{x,y,z\}.
$$

These are the *columns* of $R$, which are orthonormal.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import FloatSlider, VBox, HBox, Output
from IPython.display import display, Math

def Rx(a):
    c, s = np.cos(a), np.sin(a)
    return np.array([[1,0,0],[0,c,-s],[0,s,c]])

def Ry(a):
    c, s = np.cos(a), np.sin(a)
    return np.array([[c,0,s],[0,1,0],[-s,0,c]])

def Rz(a):
    c, s = np.cos(a), np.sin(a)
    return np.array([[c,-s,0],[s,c,0],[0,0,1]])

ax_slider = FloatSlider(description='α (x)', min=-180, max=180, step=1, value=0, continuous_update=True)
ay_slider = FloatSlider(description='β (y)', min=-180, max=180, step=1, value=0, continuous_update=True)
az_slider = FloatSlider(description='γ (z)', min=-180, max=180, step=1, value=0, continuous_update=True)

plot_out = Output()
mat_out = Output()

def update(*_):
    a, b, g = np.deg2rad([ax_slider.value, ay_slider.value, az_slider.value])
    R = Rz(g) @ Ry(b) @ Rx(a)

    with plot_out:
        plot_out.clear_output(wait=True)
        fig = plt.figure(figsize=(6,6))
        ax = fig.add_subplot(111, projection='3d')
        O = np.zeros(3)

        # Fixed laboratory axes
        for v, label in zip(np.eye(3), ['x','y','z']):
            ax.quiver(*O, *v, color='k', linewidth=2, arrow_length_ratio=0.12)
            ax.text(*(1.08*v), label, color='k', fontsize=12)

        # Rotated body axes: columns of R
        for v, color, label in zip(R.T, ['r','g','b'], ["x'","y'","z'"]):
            ax.quiver(*O, *v, color=color, linewidth=4, arrow_length_ratio=0.12)
            ax.text(*(1.08*v), label, color=color, fontsize=12)

        lim = 1.2
        ax.set(xlim=(-lim,lim), ylim=(-lim,lim), zlim=(-lim,lim))
        ax.set_box_aspect((1,1,1))
        ax.set_xlabel('X')
        ax.set_ylabel('Y')
        ax.set_zlabel('Z')
        ax.set_title('Fixed axes (black) and rotated axes (RGB)')
        ax.view_init(elev=22, azim=35)
        plt.show()

    with mat_out:
        mat_out.clear_output(wait=True)
    
        colors = ['red', 'green', 'blue']
        rows = [
            ' & '.join(
                rf'\color{{{colors[j]}}}{{{R[i,j]:+.4f}}}'
                for j in range(3)
            )
            for i in range(3)
        ]
    
        latex = r'R = \begin{pmatrix}' + r'\\'.join(rows) + r'\end{pmatrix}'
        display(Math(latex))
    
        print(f'det(R) = {np.linalg.det(R):.8f}')
        print(f'||RᵀR - I|| = {np.linalg.norm(R.T @ R - np.eye(3)):.2e}')

for s in (ax_slider, ay_slider, az_slider):
    s.observe(update, names='value')

controls = VBox([ax_slider, ay_slider, az_slider, mat_out])
display(HBox([controls, plot_out]))
update()
